In [1]:
!pip install dash -q
!pip install dash_core_components
!pip install dash-bootstrap-components
!pip install numpy
!pip install "plotly[express]" --upgrade
!pip install pandas
!pip install matplotlib
!pip install seaborn

You should consider upgrading via the '/Users/user/Desktop/bae/нод/venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/user/Desktop/bae/нод/venv/bin/python3 -m pip install --upgrade pip' command.
  Using cached dash_bootstrap_components-2.0.4-py3-none-any.whl (204 kB)
You should consider upgrading via the '/Users/user/Desktop/bae/нод/venv/bin/python3 -m pip install --upgrade pip' command.


In [23]:
import dash
import numpy as np
from dash import Dash, html, dcc, Input, Output
import dash_bootstrap_components as dbc
import plotly.express as px
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import dash_bootstrap_components as dbc

In [36]:
class GroomingDashboard:

    PALETTE = ['HotPink', 'Indigo', '#7B68EE', '#FFD700']
    FONT = 'Sans-Serif'

    def __init__(self, path: str):
        '''path - путь к csv файлу'''
        self.df = self._load_data(path)
        self.stats = self._calc_stats()
        self.instruments = sorted(self.df['инструмент'].unique().tolist())
        self.app = Dash(__name__, external_stylesheets=[dbc.themes.COSMO])
        self._setup_layout()
        self._setup_callbacks()

    def _load_data(self, path: str):
        '''читает csv и возвращает датафрейм'''
        return pd.read_csv(path)

    def _calc_stats(self):
        '''считает статистику по каждому инструменту'''
        stats = self.df.groupby('инструмент')['цена'].agg(
            mean='mean',
            median='median',
            min='min',
            max='max',
            count='count'
        ).round(0).reset_index()
        stats['mean'] = stats['mean'].astype(int)
        stats['median'] = stats['median'].astype(int)
        stats['min'] = stats['min'].astype(int)
        stats['max'] = stats['max'].astype(int)
        return stats

    def _make_stat_card(self, title, value, color):
        '''создает одну карточку со статистикой'''
        return html.Div([
            html.P(title, style={'margin': '5px', 'color': '#888',
                                 'fontSize': '13px', 'fontFamily': self.FONT}),
            html.H4(value, style={'margin': '5px', 'color': color,
                                  'fontFamily': self.FONT})
        ], style={
            'background': 'white', 'padding': '15px', 'borderRadius': '10px',
            'boxShadow': '0 2px 8px rgba(0,0,0,0.1)', #стиль хайп класс модно круто тени на карточке
            'minWidth': '140px', 'textAlign': 'center',
            'border': '1px solid #eee'
        })

    def _filter_outliers(self, df_inst):
        '''убирает выбросы''' #в основном ноутбуке с парсингом такая же логика графиков 
        mean = df_inst['цена'].mean()
        median = df_inst['цена'].median()
        q1 = df_inst['цена'].quantile(0.25)
        q3 = df_inst['цена'].quantile(0.75)
        iqr = q3 - q1
        if mean > median * 1.5:
            max_price = df_inst['цена'].max()
            return df_inst[df_inst['цена'] < max_price * 0.6], mean, median
        else:
            return df_inst[df_inst['цена'] < q3 + 1.5 * iqr], mean, median

    def _base_layout(self):
        '''базовые настройки layout для всех графиков'''
        return dict(
            plot_bgcolor='white',
            paper_bgcolor='white',
            font=dict(family=self.FONT),
            margin=dict(t=80, l=60, r=40, b=60),
        )

    def _base_axes(self):
        '''базовые настройки осей и рамка'''
        return dict(showline=True, linecolor='#ccc', mirror=True)

    def _setup_layout(self):
        '''html дэшборда'''
        options = [{'label': i, 'value': i} for i in self.instruments]

        self.app.layout = html.Div([

            html.H1('Анализ цен на инструменты для груминга',
                    style={'textAlign': 'center', 'fontFamily': self.FONT,
                           'color': '#333', 'paddingTop': '20px'}),

            html.Div([
                html.Label('Груминговый инструмент:',
                           style={'fontFamily': self.FONT, 'fontWeight': 'bold'}),
                dcc.Dropdown(
                    id='instrument-dropdown',
                    options=options,
                    value=self.instruments[0],
                    clearable=False,
                    style={'width': '400px', 'fontFamily': self.FONT}
                )
            ], style={'margin': '20px'}),

            html.Div(id='stats-cards',
                     style={'display': 'flex', 'gap': '15px',
                            'margin': '20px', 'flexWrap': 'wrap'}),

            html.Div([
                html.Div([dcc.Graph(id='price-hist')], style={'width': '50%'}),
                html.Div([dcc.Graph(id='price-scatter')], style={'width': '50%'}),
            ], style={'display': 'flex', 'gap': '10px', 'margin': '10px'}),

            html.Hr(style={'margin': '30px 20px'}),

            html.H3('Сравнение всех инструментов',
                    style={'fontFamily': self.FONT, 'margin': '20px'}),

            dcc.Graph(id='comparison-bar'),

            html.H3('Итоговая таблица затрат',
                    style={'fontFamily': self.FONT, 'margin': '20px'}),

            html.Div(id='summary-table', style={'margin': '20px'}),

        ], style={'backgroundColor': '#fafafa', 'minHeight': '100vh'})

    def _setup_callbacks(self):
        '''регистрирует все callbacks'''
        df = self.df
        stats = self.stats
        palette = self.PALETTE

        @self.app.callback(Output('stats-cards', 'children'),Input('instrument-dropdown', 'value'))
        def update_stats(selected):
            row = stats[stats['инструмент'] == selected].iloc[0]
            return [
                self._make_stat_card('Товаров', str(row['count']), '#333'),
                self._make_stat_card('Среднее', f'{row["mean"]} ₽', palette[0]),
                self._make_stat_card('Медиана', f'{row["median"]} ₽', palette[2]),
                self._make_stat_card('Мин', f'{row["min"]} ₽', '#3BB273'),
                self._make_stat_card('Макс', f'{row["max"]} ₽', '#E84855'),
            ]

        @self.app.callback(
            Output('price-hist', 'figure'),
            Input('instrument-dropdown', 'value')
        )
        def update_hist(selected):
            df_inst = df[df['инструмент'] == selected].copy()
            df_no_out, mean, median = self._filter_outliers(df_inst)

            fig = px.histogram(df_no_out, x='цена', nbins=15,
                               title=f'Распределение цен (без выбросов) — {selected}',
                               color_discrete_sequence=[palette[0]],
                               labels={'цена': 'Цена (руб.)',
                                       'count': 'Кол-во'})

            fig.update_traces(hovertemplate='Цена: %{x}<br>Кол-во: %{y}<extra></extra>')

            fig.add_vline(x=mean, line_dash='dash', line_color=palette[1])
            fig.add_vline(x=median, line_dash='dot', line_color=palette[2])

            fig.add_annotation(xref='paper', yref='paper', x=0.98, y=0.98,
                text=f'Среднее: {round(mean)} ₽<br>Медиана: {round(median)} ₽',
                showarrow=False, align='right',
                bgcolor='white', bordercolor='#ccc', borderwidth=1,
                font=dict(family=self.FONT, size=12))

            layout = self._base_layout()
            layout['xaxis_title'] = 'Цена (руб.)'
            layout['yaxis_title'] = 'Кол-во товаров'
            fig.update_layout(**layout)
            fig.update_yaxes(**self._base_axes())
            fig.update_xaxes(**self._base_axes())
            return fig

        @self.app.callback(Output('price-scatter', 'figure'), Input('instrument-dropdown', 'value'))
        def update_scatter(selected):
            df_inst = df[df['инструмент'] == selected].dropna(
                subset=['цена', 'оценка']).copy()

            def rating_cat(r):
                if r >= 4.9:
                    return '4.9+'
                elif r >= 4.7:
                    return '4.7-4.8'
                else:
                    return '4.5-4.6'

            df_inst['категория рейтинга'] = df_inst['оценка'].apply(rating_cat)
            
            fig = px.scatter(df_inst, x='оценка', y='цена',
                             color='категория рейтинга',
                             color_discrete_map={
                                 '4.5-4.6': palette[3],
                                 '4.7-4.8': palette[2],
                                 '4.9+': palette[0]
                             },
                             hover_data=['название', 'бренд'],
                             title=f'Цена vs Рейтинг — {selected}',
                             opacity=0.7)
            layout = self._base_layout()
            layout['xaxis_title'] = 'Рейтинг'
            layout['yaxis_title'] = 'Цена (руб.)'
            fig.update_layout(**layout)
            fig.update_yaxes(showline=True, linecolor='#ccc', mirror=True)
            fig.update_xaxes(showline=True, linecolor='#ccc', mirror=True)
            return fig

        @self.app.callback(Output('comparison-bar', 'figure'), Input('instrument-dropdown', 'value'))
        def update_comparison_bar(_):
            stats_sorted = stats.sort_values('median', ascending=True)
            fig = go.Figure()
            fig.add_trace(go.Bar(
                y=stats_sorted['инструмент'],
                x=stats_sorted['median'],
                name='Медиана',
                orientation='h',
                marker_color=palette[2],
                opacity=0.85
            ))
            fig.add_trace(go.Bar(
                y=stats_sorted['инструмент'],
                x=stats_sorted['mean'],
                name='Среднее',
                orientation='h',
                marker_color=palette[0],
                opacity=0.85
            ))
            layout = self._base_layout()
            layout['title'] = 'Медианная и средняя цена по инструментам'
            layout['xaxis_title'] = 'Цена (руб.)'
            layout['barmode'] = 'group'
            layout['height'] = 500
            fig.update_layout(**layout)
            fig.update_yaxes(showline=True, linecolor='#ccc', mirror=True)
            fig.update_xaxes(showline=True, linecolor='#ccc', mirror=True)
            return fig

        @self.app.callback(Output('summary-table', 'children'), Input('instrument-dropdown', 'value'))
        def update_table(_):
            rows = []
            for _, row in stats.iterrows():
                rows.append(html.Tr([
                    html.Td(row['инструмент'],
                            style={'padding': '8px', 'borderBottom': '1px solid #eee'}),
                    html.Td(f'{row["median"]} ₽',
                            style={'padding': '8px', 'borderBottom': '1px solid #eee'}),
                    html.Td(f'{row["mean"]} ₽',
                            style={'padding': '8px', 'borderBottom': '1px solid #eee'}),
                    html.Td(f'{row["min"]} ₽',
                            style={'padding': '8px', 'borderBottom': '1px solid #eee'}),
                    html.Td(f'{row["max"]} ₽',
                            style={'padding': '8px', 'borderBottom': '1px solid #eee'}),
                    html.Td(str(row['count']),
                            style={'padding': '8px', 'borderBottom': '1px solid #eee'}),
                ]))

            total_median = stats['median'].sum()
            total_mean = stats['mean'].sum()

            rows.append(html.Tr([
                html.Td('ИТОГО',
                        style={'fontWeight': 'bold', 'padding': '8px'}),
                html.Td(f'{total_median} ₽',
                        style={'fontWeight': 'bold', 'color': palette[2], 'padding': '8px'}),
                html.Td(f'{total_mean} ₽',
                        style={'fontWeight': 'bold', 'color': palette[0], 'padding': '8px'}),
                html.Td(''), html.Td(''), html.Td(''),
            ], style={'backgroundColor': '#f5f5f5'}))

            header_style = {'padding': '10px', 'textAlign': 'left', 'fontFamily': self.FONT}

            return html.Table([
                html.Thead(html.Tr([
                    html.Th('Инструмент', style=header_style),
                    html.Th('Медиана', style=header_style),
                    html.Th('Среднее', style=header_style),
                    html.Th('Мин', style=header_style),
                    html.Th('Макс', style=header_style),
                    html.Th('Товаров', style=header_style),
                ]), style={'backgroundColor': palette[0], 'color': 'white'}),
                html.Tbody(rows)
            ], style={
                'width': '100%', 'borderCollapse': 'collapse',
                'fontFamily': self.FONT, 'fontSize': '14px',
                'border': '1px solid #eee', 'borderRadius': '8px'
            })

    def run(self):
        '''запускает дашборд'''
        self.app.run(debug=False)

In [37]:
dashboard = GroomingDashboard('/Users/user/Desktop/bae/нод/проект/grooming.csv')
dashboard.run()